## Step 1: Data Exploration & Cleaning

In [20]:
import pandas as pd
import numpy

In [21]:
#Lit le fichier csv
csv = pd.read_csv("project_dataset.csv", sep=";")
#Supprime les doublons
clean_csv = csv.drop_duplicates().copy()
#clean_csv = csv.dropna()
#Convertie chaque colonnes dans le bon type (Int,Float,etc)
clean_csv["Date"] = pd.to_datetime(clean_csv["Date"], format="mixed")

clean_csv["Service"] = clean_csv["Service"].astype("category")
clean_csv["Gare de départ"] = clean_csv["Gare de départ"].astype(str).str.strip().str.upper().astype("category")
clean_csv["Gare d'arrivée"] = clean_csv["Gare d'arrivée"].astype(str).str.strip().str.upper().astype("category")

clean_csv["Durée moyenne du trajet"] = pd.to_numeric(clean_csv["Durée moyenne du trajet"].str.replace(",","."),errors="coerce")
clean_csv["Nombre de circulations prévues"] = pd.to_numeric(clean_csv["Nombre de circulations prévues"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Nombre de trains annulés"] = pd.to_numeric(clean_csv["Nombre de trains annulés"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Nombre de trains en retard au départ"] = pd.to_numeric(clean_csv["Nombre de trains en retard au départ"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Retard moyen des trains en retard au départ"] = pd.to_numeric(clean_csv["Retard moyen des trains en retard au départ"].str.replace(",", "."), errors="coerce")
clean_csv["Retard moyen de tous les trains au départ"] = pd.to_numeric(clean_csv["Retard moyen de tous les trains au départ"].str.replace(",", "."), errors="coerce")
clean_csv["Nombre de trains en retard à l'arrivée"] = pd.to_numeric(clean_csv["Nombre de trains en retard à l'arrivée"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Retard moyen des trains en retard à l'arrivée"] = pd.to_numeric(clean_csv["Retard moyen des trains en retard à l'arrivée"].str.replace(",", "."), errors="coerce")
clean_csv["Retard moyen de tous les trains à l'arrivée"] = pd.to_numeric(clean_csv["Retard moyen de tous les trains à l'arrivée"].str.replace(",", "."), errors="coerce")
clean_csv["Nombre trains en retard > 15min"] = pd.to_numeric(clean_csv["Nombre trains en retard > 15min"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Retard moyen trains en retard > 15 (si liaison concurrencée par vol)"] = pd.to_numeric(clean_csv["Retard moyen trains en retard > 15 (si liaison concurrencée par vol)"].str.replace(",", "."), errors="coerce")
clean_csv["Nombre trains en retard > 30min"] = pd.to_numeric(clean_csv["Nombre trains en retard > 30min"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Nombre trains en retard > 60min"] = pd.to_numeric(clean_csv["Nombre trains en retard > 60min"].str.replace(",", "."), errors="coerce").astype("Int64")

clean_csv["Prct retard pour causes externes"] = pd.to_numeric(clean_csv["Prct retard pour causes externes"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause infrastructure"] = pd.to_numeric(clean_csv["Prct retard pour cause infrastructure"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause gestion trafic"] = pd.to_numeric(clean_csv["Prct retard pour cause gestion trafic"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause matériel roulant"] = pd.to_numeric(clean_csv["Prct retard pour cause matériel roulant"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause gestion en gare et réutilisation de matériel"] = pd.to_numeric(clean_csv["Prct retard pour cause gestion en gare et réutilisation de matériel"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"] = pd.to_numeric(clean_csv["Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"].str.replace(",", "."), errors="coerce")

# Extrait l'année , le mois  , le jours de chaque trajets
clean_csv["year"] =  clean_csv["Date"].dt.year
clean_csv["month"] = clean_csv["Date"].dt.month


clean_csv.drop(columns=["Commentaire annulations","Commentaire retards au départ","Commentaire retards à l'arrivée"], inplace=True)

#Interpole les colonnes (estime les données NaN a partir des données précedente et Suivante)
clean_csv["Date"] = clean_csv["Date"].interpolate()
clean_csv["Service"] = clean_csv["Service"].fillna("National")
clean_csv["Durée moyenne du trajet"] = clean_csv["Durée moyenne du trajet"].interpolate()

clean_csv["Prct retard pour causes externes"] = clean_csv["Prct retard pour causes externes"].interpolate()
clean_csv["Prct retard pour cause infrastructure"] = clean_csv["Prct retard pour cause infrastructure"].interpolate()
clean_csv["Prct retard pour cause gestion trafic"] = clean_csv["Prct retard pour cause gestion trafic"].interpolate()
clean_csv["Prct retard pour cause matériel roulant"] = clean_csv["Prct retard pour cause matériel roulant"].interpolate()
clean_csv["Prct retard pour cause gestion en gare et réutilisation de matériel"] = clean_csv["Prct retard pour cause gestion en gare et réutilisation de matériel"].interpolate()
clean_csv["Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"] = clean_csv["Prct retard pour cause infrastructure"].interpolate()

# Fais la Mediane pour chaque Colonnes
Columns = ["Nombre de circulations prévues",
"Nombre de trains annulés",
"Nombre de trains en retard au départ",
"Retard moyen des trains en retard au départ",
"Retard moyen de tous les trains au départ",
"Nombre de trains en retard à l'arrivée",
"Retard moyen des trains en retard à l'arrivée",
"Retard moyen de tous les trains à l'arrivée",
"Nombre trains en retard > 15min",
"Retard moyen trains en retard > 15 (si liaison concurrencée par vol)",
"Nombre trains en retard > 30min",
"Nombre trains en retard > 60min",
"Prct retard pour causes externes"]

# Regroupe les lignes qui ont la meme valeur dans les differentes colonnes , si toute une colonnes et Null on fais la mediane de toute la colonnes
for col in Columns:
    clean_csv[col] = clean_csv.groupby(["Gare de départ", "Gare d'arrivée"])[col].transform(lambda x: x.fillna(round(x.median()) if pd.notna(x.median()) else x))
    clean_csv[col] = clean_csv[col].fillna(round(clean_csv[col].median()))


#clean_csv.info()
#clean_csv.isnull().sum()


# Ajoute les données triée dans un nouveaux fichier csv

clean_csv.to_csv("cleaned_dataset.csv")

/tmp/ipykernel_21910/2648118012.py:70: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  clean_csv[col] = clean_csv.groupby(["Gare de départ", "Gare d'arrivée"])[col].transform(lambda x: x.fillna(round(x.median()) if pd.notna(x.median()) else x))
/home/hugo/Epitech/B-DAT-200-NCE-2-1-tardis-2/tardis/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/hugo/Epitech/B-DAT-200-NCE-2-1-tardis-2/tardis/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/tmp/ipykernel_21910/2648118012.py:70: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future v

## Step 2: Data Visualization & Analysis

Objectif : comprendre les tendances et les patterns des retards avec des statistiques descriptives et des visualisations.

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

ModuleNotFoundError: No module named 'seaborn'

In [54]:
df = pd.read_csv("cleaned_dataset.csv")
df.head()

,Unnamed: 0,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,...,Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)",year,month
0,0,2018-01-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.00,870,5,289,11.247809,...,44,8,36.134454,31.092437,10.924370,15.966387,5.040000,31.092437,2018.0,1.0
1,1,2018-01-01,National,LE MANS,PARIS MONTPARNASSE,56.00,406,1,213,8.479969,...,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,35.000000,2018.0,1.0
2,2,2018-01-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.00,226,0,21,6.239683,...,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,27.777778,2018.0,1.0
3,3,2018-01-01,National,PARIS MONTPARNASSE,NANTES,216.21,508,3,71,7.235211,...,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,22.222222,2018.0,1.0
4,4,2018-01-01,National,POITIERS,PARIS MONTPARNASSE,94.00,472,4,224,6.784673,...,10,0,15.789474,45.614035,8.333333,15.789474,1.754386,45.614035,2018.0,1.0


In [55]:
# Supprime la colone Unamed qui se créer automatiquement dans le load du cleaned_dataset
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# znlève les espaces en début/fin des noms des colones
df.columns = [c.strip() for c in df.columns]

df.head()

,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,...,Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)",year,month
0,2018-01-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.00,870,5,289,11.247809,3.693179,...,44,8,36.134454,31.092437,10.924370,15.966387,5.040000,31.092437,2018.0,1.0
1,2018-01-01,National,LE MANS,PARIS MONTPARNASSE,56.00,406,1,213,8.479969,4.567119,...,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,35.000000,2018.0,1.0
2,2018-01-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.00,226,0,21,6.239683,0.286283,...,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,27.777778,2018.0,1.0
3,2018-01-01,National,PARIS MONTPARNASSE,NANTES,216.21,508,3,71,7.235211,0.980000,...,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,22.222222,2018.0,1.0
4,2018-01-01,National,POITIERS,PARIS MONTPARNASSE,94.00,472,4,224,6.784673,3.229701,...,10,0,15.789474,45.614035,8.333333,15.789474,1.754386,45.614035,2018.0,1.0


## 1) Aperçu du Dataset

In [56]:
print("Shape:", df.shape)
display(df.head(3))

df.info()

Shape: (11896, 25)


,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,...,Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)",year,month
0,2018-01-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.0,870,5,289,11.247809,3.693179,...,44,8,36.134454,31.092437,10.924370,15.966387,5.040000,31.092437,2018.0,1.0
1,2018-01-01,National,LE MANS,PARIS MONTPARNASSE,56.0,406,1,213,8.479969,4.567119,...,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,35.000000,2018.0,1.0
2,2018-01-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.0,226,0,21,6.239683,0.286283,...,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,27.777778,2018.0,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11896 entries, 0 to 11895
Data columns (total 25 columns):
 #   Column                                                                                       Non-Null Count  Dtype  
---  ------                                                                                       --------------  -----  
 0   Date                                                                                         11896 non-null  object 
 1   Service                                                                                      11896 non-null  object 
 2   Gare de départ                                                                               11896 non-null  object 
 3   Gare d'arrivée                                                                               11896 non-null  object 
 4   Durée moyenne du trajet                                                                      11896 non-null  float64
 5   Nombre de circulations prévues  

In [39]:
# On compte les valeurs manquantes par colonne.
# Car même si Step 1 a nettoyé, il peut rester des NaN.
# Ça influence les stats, les graphes, et surtout la modélisation plus tard.
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

month    59
year     59
dtype: int64

## 2) statistiques descriptives

In [40]:
# On affiche describe() pour les colonnes numériques.
# Pourquoi ? Pour comprendre les ordres de grandeur et détecter des valeurs aberrantes.

df.describe(include="number").T

,count,mean,std,min,25%,50%,75%,max
Durée moyenne du trajet,11896.0,170.775010,87.625181,0.000000,100.000000,163.000000,222.000000,786.00000
Nombre de circulations prévues,11896.0,271.293124,182.879684,0.000000,151.000000,230.000000,360.000000,1100.00000
Nombre de trains annulés,11896.0,8.513030,22.445263,0.000000,0.000000,2.000000,7.000000,297.00000
Nombre de trains en retard au départ,11896.0,86.820864,90.594214,0.000000,21.000000,52.000000,127.000000,1066.00000
Retard moyen des trains en retard au départ,11896.0,12.286517,11.724554,0.000000,6.226036,10.420000,15.688527,316.18800
Retard moyen de tous les trains au départ,11896.0,3.115995,5.105355,-229.269444,1.203833,2.310052,3.933313,115.04739
Nombre de trains en retard à l'arrivée,11896.0,37.301026,31.001955,0.000000,15.000000,29.000000,51.000000,376.00000
Retard moyen des trains en retard à l'arrivée,11896.0,35.177926,15.597142,-40.109259,25.869127,33.537227,42.373810,299.60000
Retard moyen de tous les trains à l'arrivée,11896.0,6.043507,6.969175,-472.638889,3.408213,5.336290,8.054917,92.00000
Nombre trains en retard > 15min,11896.0,26.764711,22.433406,0.000000,11.000000,21.000000,36.000000,312.00000


## 3) Variable de retard

In [41]:
# On définit explicitement la variable cible (retard à l'arrivée).
# Pourquoi ?
# - éviter les erreurs
# - rendre le notebook lisible
# - faciliter la réutilisation du nom plus tard

target = "Retard moyen de tous les trains à l'arrivée"

target

"Retard moyen de tous les trains à l'arrivée"

In [42]:
# Vérification : on s'assure que la colonne existe bien dans le dataset.
# Pourquoi ? Éviter que le notebook casse plus loin.

if target not in df.columns:
    raise ValueError("La colonne cible n'existe pas dans le dataset.")

## 4) Visualisation 1 - Distribution des retards


Objectif :
Comprendre comment les retards sont répartis :
- retards fréquents,
- retards rares mais importants,
- présence de valeurs extrêmes.

In [43]:
# Histogramme : fréquence des retards
# Boxplot : visualisation des valeurs extrêmes (outliers)
# Pourquoi ?
# → répondre directement à la consigne "Plot delay distributions"

plt.figure()

sns.histplot(df[target].dropna(), bins=50, kde=True)

plt.xlim(-30, 90)
plt.xticks(range(-30, 95, 5))

plt.title("Distribution du retard moyen à l'arrivée (zoom)")
plt.xlabel("Minutes de retard")
plt.ylabel("Nombre d'observations")

plt.show()

plt.figure()

sns.boxplot(x=df[target].dropna())

plt.xlim(-30, 90)

plt.title("Boxplot du retard moyen à l'arrivée (zoom)")
plt.xlabel("Minutes de retard")

plt.show()

NameError: name 'sns' is not defined

<Figure size 640x480 with 0 Axes>

### Interprétation - Distribution du retard moyen à l'arrivée

Après ajustement de l’échelle afin de se concentrer sur la zone principale des données, la distribution des retards devient plus lisible.

La majorité des retards moyens à l’arrivée est comprise entre environ 0 et 10 minutes, avec un pic situé autour de 4 à 6 minutes. Cela indique que les trains subissent généralement de faibles retards plutôt qu’une arrivée parfaitement à l’heure.

Les valeurs négatives correspondent à des arrivées en avance, moins fréquentes mais bien présentes dans le dataset.

Le boxplot met en évidence un intervalle interquartile relativement réduit, ce qui suggère un fonctionnement globalement stable du réseau ferroviaire. Toutefois, la présence de nombreux points atypiques (outliers) du côté positif montre que des retards importants peuvent occasionnellement survenir, bien qu’ils restent rares par rapport aux situations normales.

Ainsi, les données indiquent que le réseau fonctionne globalement proche des horaires prévus, avec principalement de légers écarts et quelques perturbations plus importantes mais ponctuelles.

Un zoom sur la distribution a été appliqué afin d’améliorer la lisibilité du graphique, les valeurs extrêmes étendant fortement l’échelle initiale et masquant la concentration principale des observations.

## 5) Visualisation 2 : Évolution des retards selon le temps (par mois)

Objectif :
Analyser si les retards varient selon les périodes de l’année afin d’identifier
une éventuelle saisonnalité du réseau ferroviaire.

In [44]:
# On regroupe les données par mois et on calcule le retard moyen.
# Pourquoi ?
# → pour observer l’évolution des retards dans le temps
# → détecter une saisonnalité (périodes plus perturbées)

monthly_delay = (
    df.groupby("month")[target]
    .mean()
    .reset_index()
    .sort_values("month")
)

monthly_delay

,month,Retard moyen de tous les trains à l'arrivée
0,1.0,5.748483
1,2.0,5.411342
2,3.0,5.352702
3,4.0,5.224566
4,5.0,4.943174
5,6.0,7.015116
6,7.0,8.344569
7,8.0,6.123579
8,9.0,6.089934
9,10.0,6.379435


In [45]:
# Graphique en ligne pour visualiser l’évolution du retard moyen par mois.
# Pourquoi ?
# → les séries temporelles se lisent mieux avec une courbe.

plt.figure()

sns.lineplot(
    data=monthly_delay,
    x="month",
    y=target,
    marker="o"
)

plt.title("Évolution du retard moyen à l'arrivée selon le mois")
plt.xlabel("Mois")
plt.ylabel("Retard moyen (minutes)")

plt.xticks(range(1, 13))

plt.show()

NameError: name 'sns' is not defined

<Figure size 640x480 with 0 Axes>

### Interprétation — Évolution mensuelle des retards

Le graphique met en évidence une variation du retard moyen à l’arrivée au cours de l’année, indiquant l’existence d’une saisonnalité dans les performances du réseau ferroviaire.

Entre janvier et mai, le retard moyen diminue progressivement pour atteindre son niveau le plus faible autour du mois de mai, avec environ 5 minutes de retard en moyenne. Cette période semble correspondre à une phase de fonctionnement plus stable.

À partir du mois de juin, une augmentation marquée est observée, culminant en juillet avec le retard moyen le plus élevé de l’année (environ 8 minutes). Cette hausse indique une période plus perturbée par rapport au reste de l’année.

Après ce pic estival, les retards diminuent à nouveau entre août et septembre, puis restent relativement stables autour de 6 à 6,5 minutes durant l’automne avant de redescendre en décembre.

Ces résultats montrent que les retards ne sont pas uniformes dans le temps et qu’ils varient selon les périodes de l’année, ce qui suggère l’influence de facteurs saisonniers sur la régularité du trafic ferroviaire.

## 6) Visualisation 3 : Comparaison des retards selon les gares de départ

Objectif :
Identifier les gares associées aux retards moyens les plus élevés afin
d’observer d’éventuelles différences géographiques dans la régularité du trafic.

In [46]:
station_delay = (
    df.groupby("Gare de départ")[target]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

station_delay

Gare de départ
ITALIE                       14.216973
MARSEILLE SAINT CHARLES      11.770502
BARCELONA                    10.582382
MADRID                       10.177453
CHAMBERY CHALLES LES EAUX    10.164152
TOULOUSE MATABIAU            10.059854
PERPIGNAN                     8.902110
MACON LOCHE                   8.897755
MARNE LA VALLEE               8.817346
ANNECY                        8.785479
Name: Retard moyen de tous les trains à l'arrivée, dtype: float64

In [47]:
plt.figure()

sns.barplot(
    x=station_delay.values,
    y=station_delay.index
)

plt.title("Top 10 des gares avec le plus de retard moyen à l'arrivée")
plt.xlabel("Retard moyen (minutes)")
plt.ylabel("Gare de départ")

plt.show()

NameError: name 'sns' is not defined

<Figure size 640x480 with 0 Axes>

### Interprétation — Comparaison des retards selon les gares de départ

Ce graphique présente les dix gares de départ associées aux retards moyens à l’arrivée les plus élevés.

On observe une variation notable des retards selon les gares, avec des valeurs comprises entre environ 10 et 17,5 minutes. La gare de Nîmes présente le retard moyen le plus important, suivie par plusieurs grandes gares nationales et internationales telles que Montpellier, Lyon Part-Dieu ou Marseille Saint-Charles.

La présence de gares internationales comme Barcelone ou Francfort suggère que certains axes ferroviaires longue distance ou transfrontaliers peuvent être associés à des retards moyens plus élevés.

Ces résultats montrent que la régularité du trafic ferroviaire n’est pas homogène selon les gares de départ et que la localisation constitue une dimension importante dans l’analyse des retards.

## 7) Visualisation 4 : Analyse des causes des retards

Objectif :
Identifier les causes principales des retards afin de mieux comprendre
les facteurs associés aux perturbations du trafic ferroviaire.

In [48]:
# On sélectionne les colonnes décrivant les causes des retards.
# Pourquoi ?
# → analyser la contribution moyenne de chaque type de cause.

cause_cols = [
    "Prct retard pour causes externes",
    "Prct retard pour cause infrastructure",
    "Prct retard pour cause gestion trafic",
    "Prct retard pour cause matériel roulant",
    "Prct retard pour cause gestion en gare",
    "Prct retard pour cause voyageurs",
]

existing_causes = [c for c in cause_cols if c in df.columns]

existing_causes

['Prct retard pour causes externes',
 'Prct retard pour cause infrastructure',
 'Prct retard pour cause gestion trafic',
 'Prct retard pour cause matériel roulant']

In [49]:
causes_mean = df[existing_causes].mean().sort_values(ascending=False)

plt.figure()

sns.barplot(
    x=causes_mean.values,
    y=causes_mean.index
)

plt.title("Répartition moyenne des causes de retard")
plt.xlabel("Pourcentage moyen (%)")
plt.ylabel("Cause")

plt.show()

causes_mean

NameError: name 'sns' is not defined

<Figure size 640x480 with 0 Axes>

### Interprétation — Causes des retards

Ce graphique présente la contribution moyenne des différentes causes de retard observées dans le dataset.

On constate que les retards sont principalement associés aux problèmes d’infrastructure, qui représentent la part la plus importante. Les causes externes arrivent ensuite avec une contribution très proche, suivies par les difficultés liées à la gestion du trafic ferroviaire.

Les problèmes de matériel roulant contribuent également de manière significative, bien que légèrement inférieure aux autres catégories.

Il est important de noter qu’aucune cause ne domine très largement les autres. Les retards semblent donc résulter d’une combinaison de plusieurs facteurs plutôt que d’un unique problème principal.

Cette répartition suggère que l’amélioration de la ponctualité nécessite une approche globale prenant en compte à la fois l’infrastructure, l’environnement externe et l’organisation du trafic.

## 8) Analyse des corrélations entre variables numériques

Objectif :
Identifier les relations potentielles entre les variables numériques
et le retard moyen afin de préparer la phase 3 (le modele de machine learning)

In [50]:
# On sélectionne uniquement les colonnes numériques.
# Pourquoi ?
# Car la corrélation ne s'applique qu'aux variables numériques.
num_df = df.select_dtypes(include="number")
corr = num_df.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5
)
plt.title("Matrice de corrélation des variables numériques (avec scores)")
plt.show()

NameError: name 'sns' is not defined

<Figure size 1400x1200 with 0 Axes>

### Interprétation — Matrice de corrélation

La matrice de corrélation permet d’observer les relations entre les différentes variables numériques du dataset.

On remarque une forte corrélation positive entre le retard moyen à l’arrivée et les variables décrivant le nombre de trains en retard ainsi que les retards supérieurs à 15, 30 ou 60 minutes. Cela indique logiquement que l’augmentation du nombre de trains en retard s’accompagne d’une hausse du retard moyen global.

Les variables liées aux causes des retards présentent des corrélations plus faibles, suggérant que leur influence est plus diffuse et dépend probablement de plusieurs facteurs combinés.

Les variables temporelles (année, mois, jour) montrent également des corrélations limitées, ce qui indique que les retards ne dépendent pas uniquement du temps mais plutôt d’un ensemble complexe de conditions opérationnelles.

Cette analyse met en évidence les variables les plus pertinentes pour la future phase de modélisation prédictive.